# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [58]:
import warnings
warnings.filterwarnings('ignore')

In [59]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = userdata.get('HUGGINGFACEHUB_API_TOKEN')


In [60]:
import pandas as pd
df = pd.read_csv('/content/Data.csv')

In [61]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [62]:
!pip install langchain langchain-core langchain-openai langchain-community langchain-classic

In [63]:
from langchain_openai import ChatOpenAI        # ✅ 1.0.3
from langchain_core.prompts import ChatPromptTemplate  # ✅ From langchain-core 1.0.5
from langchain_classic.chains import LLMChain  # ✅ Legacy (install if missing)

In [64]:
#Replace None by your own value and justify
llm = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=0.7)


In [65]:
prompt = ChatPromptTemplate.from_template(
       """
    You are an expert product recommendation assistant.

    Write a short recommendation article about the following product:

    Product: {product}

    Include:
    - Main strengths
    - Best use case
    - Overall recommendation
    """
)

In [66]:

chain = LLMChain(llm=llm, prompt=prompt)

In [67]:
product = df.loc[0, "Product"]
chain.run(product)

"Queen Size Sheet Set is a highly recommended product for those looking to enhance the comfort and style of their bedroom. \n\nThe main strengths of this sheet set include its luxurious feel, durability, and perfect fit for a queen size bed. Made from high-quality materials, these sheets are soft to the touch and provide a cozy night's sleep. Additionally, they are designed to withstand multiple washes without losing their color or shape, making them a long-lasting investment.\n\nThe best use case for the Queen Size Sheet Set is for anyone with a queen size bed looking to upgrade their bedding. Whether you're redecorating your bedroom or simply want to treat yourself to a better night's sleep, these sheets are a perfect choice. They are also great for guest rooms, ensuring that your visitors have a comfortable stay.\n\nOverall, I highly recommend the Queen Size Sheet Set for anyone in need of new bedding. With its high-quality materials, luxurious feel, and perfect fit for a queen size

## SimpleSequentialChain

In [68]:
from langchain_classic.chains import SimpleSequentialChain

In [69]:
llm = ChatOpenAI(api_key=OPENAI_API_KEY,temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    """
    Write a short and engaging product description for the following product:

    Product: {product}
    """
)

# Chain 1
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt
)



In [70]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    """
    Based on the following product description:

    {text}

    Write a short marketing recommendation explaining
    why customers should buy this product.
    """
)

# chain 2
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt
)

In [71]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [72]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Transform your bedroom into a luxurious oasis with our Queen Size Sheet Set. Made from soft and breathable materials, these sheets are perfect for a restful night's sleep. With a variety of colors to choose from, you can easily match them with your existing decor. Upgrade your bedding with this cozy and stylish sheet set today!
Customers should buy this Queen Size Sheet Set because it offers a luxurious and comfortable sleeping experience. The soft and breathable materials ensure a restful night's sleep, while the variety of colors allows for easy coordination with existing decor. By upgrading to this cozy and stylish sheet set, customers can transform their bedroom into a relaxing oasis and elevate their bedding to a new level of comfort and style.

> Finished chain.


"Customers should buy this Queen Size Sheet Set because it offers a luxurious and comfortable sleeping experience. The soft and breathable materials ensure a restful night's sleep, while the variety of colors allows for easy coordination with existing decor. By upgrading to this cozy and stylish sheet set, customers can transform their bedroom into a relaxing oasis and elevate their bedding to a new level of comfort and style."

**Repeat the above twice for different products**

## SequentialChain

In [73]:
from langchain_classic.chains import SequentialChain

In [74]:
llm = ChatOpenAI(
    temperature=0.9,
    api_key=OPENAI_API_KEY
)

first_prompt = ChatPromptTemplate.from_template(
    """
    Translate the following customer review into English:

    Review: {review}
    """
)

chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    output_key="translated_review"
)

In [75]:
second_prompt = ChatPromptTemplate.from_template(
    """
    Summarize the following customer review in 2 short sentences:

    Review: {translated_review}
    """
)

chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
)

In [76]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    """
    Translate the following summary into German:

    Summary: {summary}
    """
)

# chain 3: input=summary and output=translated_summary
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="translated_summary"
)

In [77]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    """
    Based on the following review summary in English:

    {summary}

    And its German translation:

    {translated_summary}

    Write a short and polite follow-up response to the customer.
    """
)

chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key="follow_up_message"
)

In [78]:
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review"],
    output_variables=[
        "translated_review",
        "summary",
        "translated_summary",
        "follow_up_message"
    ],
    verbose=True
)

In [79]:
review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'translated_review': "Review: I find the taste mediocre. The foam doesn't hold, it's weird. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?",
 'summary': 'The customer found the taste of the product mediocre and noted that the foam did not hold well. They suspected that the product may have been an old batch or counterfeit compared to their usual purchase in stores.',
 'translated_summary': 'Zusammenfassung: Der Kunde fand den Geschmack des Produkts mittelmäßig und stellte fest, dass der Schaum nicht gut hielt. Sie vermuteten, dass es sich bei dem Produkt möglicherweise um eine alte Charge oder eine Fälschung im Vergleich zu ihrem üblichen Einkauf im Geschäft handeln könnte.',
 'follow_up_message': 'Dear valued customer,\n\nThank you for sharing your feedback with us ab

**Repeat the above twice for different products or reviews**

## Router Chain

In [80]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [81]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "history",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [82]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate  # Updated path for prompts (core package)

In [83]:
llm = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=0.9)

In [84]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [85]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [86]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [87]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [88]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [89]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect absorber of radiation at all wavelengths. A black body absorbs all incoming radiation and emits radiation according to its temperature. The distribution of this radiation is described by Planck's law, which shows that the intensity of the radiation emitted varies with the wavelength and temperature of the object. This phenomenon is important in understanding various aspects of physics, including thermal radiation and the behavior of stars."

In [90]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [91]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA carries the genetic information that controls all of the functions and characteristics of an organism. DNA serves as the blueprint for building and maintaining an organism, determining traits such as hair color, eye color, and susceptibility to certain diseases. Additionally, DNA is necessary for cell replication and growth, as it carries the instructions for making proteins essential for cellular function. Therefore, having DNA in every cell ensures that all cells in an organism have the necessary information to perform their specific functions and maintain the overall health and function of the organism as a whole.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [93]:
# LLMChain - different product
product = df.loc[1, "Product"]
chain.run(product)

# SimpleSequentialChain - different producT
product = df.loc[2, "Product"]
overall_simple_chain.run(product)

# SequentialChain - different review
review = df.loc[2, "Review"]
overall_chain.invoke({
    "review": review
})





> Entering new MultiPromptChain chain...
None: {'input': 'Waterproof Phone Pouch'}
> Finished chain.


> Entering new SimpleSequentialChain chain...
Experience the ultimate in comfort and luxury with our top-of-the-line air mattress. Crafted with premium materials and innovative technology, this mattress provides the perfect balance of support and cushioning for a restful night's sleep. Say goodbye to uncomfortable nights and hello to the luxurious sleep you deserve. Upgrade your sleeping experience with our luxury air mattress today.
Customers should buy this top-of-the-line air mattress for the ultimate sleep experience. Crafted with premium materials and innovative technology, this mattress offers the perfect balance of support and cushioning for a restful night's sleep. Say goodbye to uncomfortable nights and hello to the luxurious sleep you deserve. Invest in your sleep and upgrade to this luxury air mattress today for a truly rejuvenating night's rest.

> Finished chain.


> En

{'review': "This mattress had a small hole in the top of it (took forever to find where it was), and the patches that they provide did not work, maybe because it's the top of the mattress where it's kind of like fabric and a patch won't stick. Maybe I got unlucky with a defective mattress, but where's quality assurance for this company? That flat out should not happen. Emphasis on flat. Cause that's what the mattress was. Seriously horrible experience, ruined my friend's stay with me. Then they make you ship it back instead of just providing a refund, which is also super annoying to pack up an air mattress and take it to the UPS store. This company is the worst, and this mattress is the worst.",
 'translated_review': "Review: This mattress had a small hole at the top (it took forever to find where it was), and the patches provided did not work, maybe because it's the fabric-like top of the mattress where a patch won't stick. Maybe I was unlucky with a defective mattress, but where is t

In [94]:
# Router Chain - different questions

chain.run("Why do astronauts feel weightless in space?")




> Entering new MultiPromptChain chain...
physics: {'input': 'Why do astronauts feel weightless in space?'}
> Finished chain.


'Astronauts feel weightless in space because they are in a state of free fall. While in orbit around Earth, they are constantly falling towards the planet due to gravity, but their high speed prevents them from actually hitting the surface. This creates the sensation of weightlessness because there is no solid surface pushing back against their bodies.'

In [95]:

chain.run("Explain recursion in simple terms.")




> Entering new MultiPromptChain chain...
computer science: {'input': 'Explain recursion in simple terms.'}
> Finished chain.


"Recursion is a programming technique where a function calls itself in order to solve a problem. It's like a loop, but instead of repeating a set of instructions, the function calls itself with a different input each time until a base case is reached and the function stops calling itself. Recursion is often used when a problem can be broken down into smaller, similar subproblems. It can be a powerful tool for solving complex problems in a more elegant and concise way."

In [96]:

chain.run("Why did the Roman Empire collapse?")




> Entering new MultiPromptChain chain...
history: {'input': 'Why did the Roman Empire collapse?'}
> Finished chain.


"The collapse of the Roman Empire is a complex and multifaceted event that is often attributed to a combination of internal and external factors. Some possible reasons for the collapse of the Roman Empire include:\n\n1. Economic troubles: The Roman Empire faced economic challenges such as overreliance on slave labor, inflation, high taxes, and a decline in trade. These economic issues weakened the empire's financial stability and ability to support its vast territories.\n\n2. Political instability: The Roman Empire experienced frequent changes in leadership, civil wars, and corruption within the government. This political instability weakened the empire's ability to govern effectively and maintain control over its territories.\n\n3. Military defeats: The Roman Empire faced external threats from invading barbarian tribes, such as the Visigoths and Vandals, as well as the Persian Empire. These military defeats weakened the empire's defenses and territorial integrity.\n\n4. Social unrest:

In [97]:

chain.run("How do plants create energy from sunlight?")



> Entering new MultiPromptChain chain...
biology: {'input': 'How do plants create energy from sunlight?'}
> Finished chain.


'Plants create energy from sunlight through a process called photosynthesis. During photosynthesis, plants use chlorophyll, a pigment found in their chloroplasts, to capture sunlight energy. This energy is used to convert carbon dioxide from the air and water from the soil into glucose (a type of sugar) and oxygen. The glucose is then used by the plant as a source of energy for growth and maintenance, while the oxygen is released into the atmosphere as a byproduct. \n\nOverall, the equation for photosynthesis is:\n6CO2 + 6H2O + light energy → C6H12O6 + 6O2\n\nThis process is essential for all life on Earth, as plants are the primary producers that provide energy for the rest of the food chain. Additionally, photosynthesis plays a crucial role in regulating atmospheric carbon dioxide levels and producing oxygen, which is necessary for aerobic respiration in animals and many other organisms.'